### Funciona para Nubank para as tabelas de crédito pessoal e de cartão. 

> Ainda é interessante deixar isso automático, até na definição de página


O que falta?
* corrija algoritmo para reconhecer - - - 
* coloque também para reconhecer CP
* melhorar extração de trimestres ( deixar melhor )

Desafios:
* tirar a necessidade de colocar a página específica
* capacidade de extrair qualquer planilha contida no PDF

In [58]:
# Por algum motivo, não pega perda esperada ou exposição bruta. 
# Há um problema oculto no código ou na forma como está escrito

# Colocar como CP ou Crédito deveria ser mais simplificado

In [59]:
# Nossas opções:
# Nubank, Inter, Nubank, BB, Santander, Itaú ( todos tem textos selecionáveis ). Resultado é definição de uma janela

# Ingerir PDF
import pdfplumber
import pandas as pd
import re
import numpy as np
import re

## 14. Extração de dados para Empréstimo a Clientes 

In [60]:
# para extração de CP ( cuidado com a página )
# 1. c) Provisão para perdas de crédito - por qualidade de crédito vs. estágios

# para extração de Crédito ( cuidado com a página )
# 1. c) Provisão para perdas de crédito - por qualidade de crédito vs. estágios

# 4T25 mudou o título para: c) Perda esperada de crédito - por qualidade de crédito vs. estágios

In [61]:
# Refere-se a seção 14

with pdfplumber.open("Demonstrações Financeiras 4T25.pdf") as pdf:
    page = pdf.pages[30]  
    text = page.extract_text()

# Remove espaços em branco extras de cada linha ( pré-processamento )
text = "\n".join(line.strip() for line in text.splitlines())

# Define início ( start ) e fim ( end ) do trecho a ser extraído. 
start = text.find("c) Perda esperada de crédito - por qualidade de crédito vs. estágios")
end = text.find("Total", start)

trecho = text[start:end]
print(trecho)

# divide em linhas para facilitar a criação do dataframe
linhas = [l.strip() for l in trecho.splitlines() if l.strip()]


c) Perda esperada de crédito - por qualidade de crédito vs. estágios
2025 2024
Perda Índice de Perda Índice de
Exposição Exposição
% esperada % cobertura % esperada % cobertura
bruta bruta
de crédito (%) de crédito (%)
Forte (PD < 5%) 3.401.763 31,2% 41.731 2,8% 1,2% 1.954.790 31,9% 19.761 2,4% 1,0%
Estágio 1 3.357.159 98,7% 41.546 99,6% 1,2% 1.883.302 96,3% 18.678 94,5% 1,0%
Estágio 2 44.604 1,3% 185 0,4% 0,4% 71.488 3,7% 1.083 5,5% 1,5%
Satisfatório (5% ≤ PD ≤ 20%) 3.756.036 34,4% 206.811 13,8% 5,5% 2.101.425 34,4% 113.253 14,3% 5,4%
Estágio 1 3.683.259 98,1% 203.933 98,6% 5,5% 1.855.922 88,3% 97.439 86,0% 5,3%
Estágio 2 72.777 1,9% 2.878 1,4% 4,0% 245.503 11,7% 15.814 14,0% 6,4%
Risco maior (PD > 20%) 3.757.654 34,4% 1.245.453 83,4% 33,1% 2.060.240 33,7% 661.556 83,3% 32,1%
Estágio 1 1.668.016 44,4% 222.137 17,8% 13,3% 989.134 48,0% 123.189 18,6% 12,5%
Estágio 2 1.410.063 37,5% 566.422 45,5% 40,2% 737.425 35,8% 308.123 46,6% 41,8%
Estágio 3 679.575 18,1% 456.894 36,7% 67,2% 333.681 

### Retirando elementos desnecessários da extração

In [62]:
def limpar_linhas_estagio(linhas):

    resultado = []
    ignorar_ate_estagio = False

    PD_HEADERS = ("Forte", "Satisfatório", "Risco maior") # Define título

    for linha in linhas:

        if linha.startswith(PD_HEADERS): # ignora tudo, menos estágios
            ignorar_ate_estagio = True
            continue

        if ignorar_ate_estagio:
            if linha.startswith("Estágio"): # Define sessão que não deve ser excluída
                ignorar_ate_estagio = False
            else:
                continue

        if not linha.startswith("Estágio"):
            continue

        tokens = linha.split()

        # remove porcentagens
        tokens_sem_percent = [t for t in tokens if "%" not in t]

        # ignora "Estágio" e o número do estágio
        tokens_dados = tokens_sem_percent[2:]

        numeros = [
            t for t in tokens_dados
            if re.fullmatch(r"\d{1,3}(?:\.\d{3})*|–", t)
        ]
        # --------------------------------

        estagio = " ".join(tokens_sem_percent[:2])  # Estágio 1 / 2 / 3

        resultado.append(
            " ".join([estagio] + numeros)
        )

    return resultado



In [63]:
# há apenas um erro: você deve ensinar o código a lidar com -. Está dando como nulo 4T25 para estágio 2 justamente pelo fato que está pegando
# o que está embaixo


In [64]:
# será que deixando o título dos headers deixa mais limpo o código?
linhas_limpa = limpar_linhas_estagio(linhas)

for l in linhas_limpa:
    print(l)

Estágio 1 3.357.159 41.546 1.883.302 18.678
Estágio 2 44.604 185 71.488 1.083
Estágio 1 3.683.259 203.933 1.855.922 97.439
Estágio 2 72.777 2.878 245.503 15.814
Estágio 1 1.668.016 222.137 989.134 123.189
Estágio 2 1.410.063 566.422 737.425 308.123
Estágio 3 679.575 456.894 333.681 230.244


### Extração do trimestre

In [65]:

from datetime import datetime

def datas_para_trimestres(texto):
    
    # 1️⃣ Primeiro tenta pegar datas completas
    datas_completas = re.findall(r"\d{2}/\d{2}/\d{4}", texto)
    
    # 2️⃣ Depois pega anos isolados (4 dígitos)
    anos_isolados = re.findall(r"\b\d{4}\b", texto)

    trimestres = []

    # 🔹 Caso existam datas completas
    for d in datas_completas:
        dt = datetime.strptime(d, "%d/%m/%Y")
        trimestre = (dt.month - 1) // 3 + 1
        ano = str(dt.year)[-2:]
        trimestres.append(f"{trimestre}T{ano}")

    # 🔹 Caso NÃO existam datas completas mas existam anos isolados
    if not datas_completas and anos_isolados:
        for ano in anos_isolados:
            dt = datetime.strptime(ano, "%Y")
            ano_formatado = str(dt.year)[-2:]
            trimestres.append(f"4T{ano_formatado}")  # regra especial

    return trimestres

In [66]:
print(datas_para_trimestres(trecho))

['4T25', '4T24']


### Padrão fixo para colunas do dataframe

Vou ter que mudar isso, já que, caso haja mais elementos no dataframe, isso pode quebrar

In [67]:
# por que agora e não antes? Por que definir estágio padrão? Isso vai quebrar o código

PD_PADRAO = [
    "Forte (PD < 5%)",
    "Forte (PD < 5%)",
    "Satisfatório (5% ≤ PD ≤ 20%)",
    "Satisfatório (5% ≤ PD ≤ 20%)",
    "Risco maior (PD > 20%)",
    "Risco maior (PD > 20%)",
    "Risco maior (PD > 20%)",
]

ESTAGIO_PADRAO = [1, 2, 1, 2, 1, 2, 3]

### Para as colunas de Exposição Bruta e PE

In [68]:
def parse_linha_estagio(linha):
    partes = linha.split()
    estagio = int(partes[1])

    numeros = partes[2:]

    def conv(x):
        if x == "–":
            return None
        return float(x.replace(".", "").replace(",", "."))

    numeros = [conv(x) for x in numeros]

    return {
        "estagio": estagio,
        "exp_bruta_atual": numeros[0] if len(numeros) > 0 else None,
        "pe_atual": numeros[1] if len(numeros) > 1 else None,
        "exp_bruta_anterior": numeros[2] if len(numeros) > 2 else None,
        "pe_anterior": numeros[3] if len(numeros) > 3 else None,
    }


In [69]:
# há PD_PADRAO para pegar o padrão do dataframe de PD ( aquele que eu disse que dá pra quebrar ). Vamos manter assim e
# testar 

# Preciso do produto de Crédito

def construir_dataframe(
    linhas_estagio,
    texto_pdf,
    banco="Nubank",
    produto="CP"
):

    trimestre_atual, trimestre_anterior = datas_para_trimestres(texto_pdf)

    registros = []

    for i, linha in enumerate(linhas_estagio):

        dados = parse_linha_estagio(linha)

        # Linha do trimestre atual
        registros.append({
            "ano": trimestre_atual,
            "banco": banco.lower(),
            "produto": produto,
            "PD": PD_PADRAO[i],
            "Estágio": ESTAGIO_PADRAO[i],
            "Exposição Bruta": dados["exp_bruta_atual"],
            "Perda Esperada": dados["pe_atual"]
        })

        # Linha do trimestre anterior
        registros.append({
            "ano": trimestre_anterior,
            "banco": banco.lower(),
            "produto": produto,
            "PD": PD_PADRAO[i],
            "Estágio": ESTAGIO_PADRAO[i],
            "Exposição Bruta": dados["exp_bruta_anterior"],
            "Perda Esperada": dados["pe_anterior"]
        })

    df = pd.DataFrame(registros)

    return df


In [70]:
# melt transforma colunas em linhas para exposição bruta

df_final = construir_dataframe(
    linhas_estagio=linhas_limpa,
    texto_pdf=trecho,
    banco="Nubank",
    produto="CP"
)

df_final

,ano,banco,produto,PD,Estágio,Exposição Bruta,Perda Esperada
0,4T25,nubank,CP,Forte (PD < 5%),1,3357159.0,41546.0
1,4T24,nubank,CP,Forte (PD < 5%),1,1883302.0,18678.0
2,4T25,nubank,CP,Forte (PD < 5%),2,44604.0,185.0
3,4T24,nubank,CP,Forte (PD < 5%),2,71488.0,1083.0
4,4T25,nubank,CP,Satisfatório (5% ≤ PD ≤ 20%),1,3683259.0,203933.0
5,4T24,nubank,CP,Satisfatório (5% ≤ PD ≤ 20%),1,1855922.0,97439.0
6,4T25,nubank,CP,Satisfatório (5% ≤ PD ≤ 20%),2,72777.0,2878.0
7,4T24,nubank,CP,Satisfatório (5% ≤ PD ≤ 20%),2,245503.0,15814.0
8,4T25,nubank,CP,Risco maior (PD > 20%),1,1668016.0,222137.0
9,4T24,nubank,CP,Risco maior (PD > 20%),1,989134.0,123189.0


### 📊 Passar para o Excel
> O que muda? Agora, temos uma nova coluna a ser preenchida: a coluna de PD

In [71]:
import openpyxl
from openpyxl import load_workbook

In [72]:
# PD é a nossa coluna D
# Produto é a coluna E
# Tipo é a coluna B
# Estágio é a coluna C

In [73]:

wb = load_workbook("Excel - Dados bancários.xlsx")
sheet = wb["Nubank"]

anos_df = df_final["ano"].unique()

# =====================
# Garante anos (linha 5 e 29)
# =====================

def garantir_anos(linha_base):
    anos_existentes = {
        str(sheet.cell(row=linha_base, column=c).value).strip()
        for c in range(1, sheet.max_column + 1)
    }

    for ano in anos_df:
        if str(ano) not in anos_existentes:
            sheet.cell(row=linha_base, column=sheet.max_column + 1, value=ano)

garantir_anos(5)
garantir_anos(29)

# =====================
# Funções auxiliares
# =====================

def encontrar_coluna(ano, linha_base):
    for c in range(1, sheet.max_column + 1):
        if str(sheet.cell(row=linha_base, column=c).value).strip() == str(ano):
            return c
    return None

def encontrar_linha(tipo, estagio, pd_val, produto, linha_inicio):
    for r in range(linha_inicio, sheet.max_row + 1):
        if (
            str(sheet.cell(r, 2).value).strip() == tipo and
            str(sheet.cell(r, 3).value).strip() == estagio and
            str(sheet.cell(r, 4).value).strip() == pd_val and
            str(sheet.cell(r, 5).value).strip() == produto
        ):
            return r
    return None

# =====================
# Preenchimento
# =====================

for _, row in df_final.iterrows():

    estagio = f"Estágio {row['Estágio']}"
    produto = row["produto"]
    pd_val = row["PD"]
    ano = row["ano"]

    for tipo, linha_base, linha_inicio in [
        ("Exposição Bruta", 5, 1),
        ("Perda Esperada", 29, 29),
    ]:

        valor = row[tipo]
        if pd.isna(valor):
            continue

        col = encontrar_coluna(ano, linha_base)
        lin = encontrar_linha(tipo, estagio, pd_val, produto, linha_inicio)

        if lin and col:
            sheet.cell(row=lin, column=col, value=float(valor))


wb.save("Excel - Dados bancários.xlsx")